In [1]:
from pathlib import Path
import sys

current = Path.cwd()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (candidate / "src" / "drift_lab").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing src/drift_lab")

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Current directory:", current)
print("Project root:", PROJECT_ROOT)
print("Src dir:", SRC_DIR)
print("Src exists:", SRC_DIR.exists())

Current directory: D:\Projects\Detecting-and-Adapting-to-Concept-Drift-in-Time-Series-Forecasting\experiments\run\detection
Project root: D:\Projects\Detecting-and-Adapting-to-Concept-Drift-in-Time-Series-Forecasting
Src dir: D:\Projects\Detecting-and-Adapting-to-Concept-Drift-in-Time-Series-Forecasting\src
Src exists: True


In [2]:
from itertools import product

import pandas as pd

from drift_lab.config import SEEDS
from drift_lab.detection.adwin import ADWINDetector
from drift_lab.detection.kswin import KSWINDetector
from drift_lab.detection.page_hinkley import PageHinkleyDetector
from drift_lab.evaluation.evaluation import evaluate_detections
from drift_lab.synthetic.generator import make_series

In [3]:
KINDS = ("none", "sudden", "gradual", "recurring")

N = 20_000
NOISE = 1.0

SAMPLES_PER_YEAR = 48 * 365
FALSE_ALARM_BUDGET_PER_YEAR = 2.0

print("Seeds:", SEEDS)
print("Drift types:", KINDS)
print("N:", N)
print("False-alarm budget/year:", FALSE_ALARM_BUDGET_PER_YEAR)

Seeds: (1, 2, 3, 4, 5)
Drift types: ('none', 'sudden', 'gradual', 'recurring')
N: 20000
False-alarm budget/year: 2.0


In [4]:
# ADWIN: 7 configurations
ADWIN_GRID = [
    0.0001,
    0.00025,
    0.0005,
    0.00075,
    0.001,
    0.0015,
    0.002,
]

# KSWIN: 6 × 2 × 7 = 84 configurations
KSWIN_ALPHA_GRID = [
    0.0005,
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
]

KSWIN_WINDOW_GRID = [
    300,
    450,
]

KSWIN_STAT_GRID = [
    40,
    44,
    48,
    50,
    55,
    60,
    65,
]

# Page-Hinkley: 6 × 3 × 6 = 108 configurations
PH_DELTA_GRID = [
    0.0005,
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
]

PH_MIN_INSTANCES_GRID = [
    20,
    30,
    50,
]

PH_THRESHOLD_GRID = [
    200.0,
    300.0,
    400.0,
    500.0,
    750.0,
    1000.0,
]

In [5]:
print("ADWIN configs:", len(ADWIN_GRID))

kswin_configs = (
    len(KSWIN_ALPHA_GRID)
    * len(KSWIN_WINDOW_GRID)
    * len(KSWIN_STAT_GRID)
)

ph_configs = (
    len(PH_DELTA_GRID)
    * len(PH_MIN_INSTANCES_GRID)
    * len(PH_THRESHOLD_GRID)
)

print("KSWIN configs:", kswin_configs)
print("Page-Hinkley configs:", ph_configs)

TOTAL_CONFIGS = (
    len(ADWIN_GRID)
    + kswin_configs
    + ph_configs
)

TOTAL_RUNS = TOTAL_CONFIGS * len(KINDS) * len(SEEDS)

print("Total configs:", TOTAL_CONFIGS)
print("Total runs:", TOTAL_RUNS)

ADWIN configs: 7
KSWIN configs: 84
Page-Hinkley configs: 108
Total configs: 199
Total runs: 3980


In [6]:
def false_alarms_per_year(
    n_false_alarms: int,
    n_observations: int,
) -> float:
    simulated_years = n_observations / SAMPLES_PER_YEAR
    return float(n_false_alarms / simulated_years)

In [7]:
def evaluate_candidate(
    detector,
    detector_name: str,
    config: dict,
):
    rows = []

    for kind in KINDS:
        for seed in SEEDS:
            series, true_changepoints = make_series(
                kind=kind,
                n=N,
                noise=NOISE,
                seed=seed,
            )

            detected = detector.detect(series)

            evaluation = evaluate_detections(
                detected_changepoints=detected,
                true_changepoints=true_changepoints,
                n_observations=len(series),
                drift_type=kind,
            )

            n_false = len(evaluation["false_alarm_indices"])

            fa_year = false_alarms_per_year(
                n_false_alarms=n_false,
                n_observations=len(series),
            )

            rows.append(
                {
                    "method": detector_name,
                    "config": str(config),
                    "drift_type": kind,
                    "seed": seed,
                    "n_detections": len(detected),
                    "n_false_alarms": n_false,
                    "false_alarms_per_year": fa_year,
                    "detection_delay": evaluation["detection_delay"],
                    "missed_detections": evaluation["missed_detections"],
                }
            )

    return rows

In [8]:
print(evaluate_candidate)

<function evaluate_candidate at 0x0000021D4D9C0040>


In [9]:
test_adwin = ADWINDetector(delta=0.00075)

adwin_test_rows = evaluate_candidate(
    detector=test_adwin,
    detector_name="adwin",
    config={"delta": 0.00075},
)

adwin_test_df = pd.DataFrame(adwin_test_rows)

print("Shape:", adwin_test_df.shape)

adwin_test_df

Shape: (20, 9)


,method,config,drift_type,seed,n_detections,n_false_alarms,false_alarms_per_year,detection_delay,missed_detections
0,adwin,{'delta': 0.00075},none,1,0,0,0.000,NaN,0
1,adwin,{'delta': 0.00075},none,2,0,0,0.000,NaN,0
2,adwin,{'delta': 0.00075},none,3,0,0,0.000,NaN,0
3,adwin,{'delta': 0.00075},none,4,0,0,0.000,NaN,0
4,adwin,{'delta': 0.00075},none,5,0,0,0.000,NaN,0
5,adwin,{'delta': 0.00075},sudden,1,1,0,0.000,23.0,0
6,adwin,{'delta': 0.00075},sudden,2,1,0,0.000,23.0,0
7,adwin,{'delta': 0.00075},sudden,3,1,0,0.000,23.0,0
8,adwin,{'delta': 0.00075},sudden,4,1,0,0.000,23.0,0
9,adwin,{'delta': 0.00075},sudden,5,1,0,0.000,23.0,0


In [10]:
test_kswin = KSWINDetector(
    alpha=0.001,
    window_size=450,
    stat_size=55,
    seed=42,
)

kswin_test_config = {
    "alpha": 0.001,
    "window_size": 450,
    "stat_size": 55,
    "seed": 42,
}

kswin_test_rows = evaluate_candidate(
    detector=test_kswin,
    detector_name="kswin",
    config=kswin_test_config,
)

kswin_test_df = pd.DataFrame(kswin_test_rows)

print("Shape:", kswin_test_df.shape)

kswin_test_df

Shape: (20, 9)


,method,config,drift_type,seed,n_detections,n_false_alarms,false_alarms_per_year,detection_delay,missed_detections
0,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",none,1,0,0,0.000,NaN,0
1,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",none,2,0,0,0.000,NaN,0
2,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",none,3,0,0,0.000,NaN,0
3,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",none,4,0,0,0.000,NaN,0
4,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",none,5,0,0,0.000,NaN,0
5,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",sudden,1,2,1,0.876,31.0,0
6,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",sudden,2,1,0,0.000,31.0,0
7,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",sudden,3,1,0,0.000,28.0,0
8,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",sudden,4,1,0,0.000,30.0,0
9,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",sudden,5,2,1,0.876,23.0,0


In [11]:
kswin_test_df.groupby("drift_type").agg(
    mean_delay=("detection_delay", "mean"),
    mean_false_alarms_per_year=("false_alarms_per_year", "mean"),
    total_missed=("missed_detections", "sum"),
    mean_detections=("n_detections", "mean"),
)

,mean_delay,mean_false_alarms_per_year,total_missed,mean_detections
drift_type,,,,
gradual,350.2,0.8760,0,2.0
none,NaN,0.0000,0,0.0
recurring,20.2,0.1752,0,2.2
sudden,28.6,0.3504,0,1.4


In [12]:
search_rows = []
completed = 0

ADWIN_TOTAL = len(ADWIN_GRID)
KSWIN_TOTAL = (
    len(KSWIN_ALPHA_GRID)
    * len(KSWIN_WINDOW_GRID)
    * len(KSWIN_STAT_GRID)
)

print("ADWIN configs:", ADWIN_TOTAL)
print("KSWIN configs:", KSWIN_TOTAL)
print("Current search total:", ADWIN_TOTAL + KSWIN_TOTAL)

ADWIN configs: 7
KSWIN configs: 84
Current search total: 91


In [13]:
for delta in ADWIN_GRID:
    detector = ADWINDetector(delta=delta)

    config = {
        "delta": delta,
    }

    rows = evaluate_candidate(
        detector=detector,
        detector_name="adwin",
        config=config,
    )

    search_rows.extend(rows)

    completed += 1

    print(
        f"[{completed}/91] "
        f"ADWIN delta={delta}"
    )

[1/91] ADWIN delta=0.0001
[2/91] ADWIN delta=0.00025
[3/91] ADWIN delta=0.0005
[4/91] ADWIN delta=0.00075
[5/91] ADWIN delta=0.001
[6/91] ADWIN delta=0.0015
[7/91] ADWIN delta=0.002


In [14]:
for alpha, window_size, stat_size in product(
    KSWIN_ALPHA_GRID,
    KSWIN_WINDOW_GRID,
    KSWIN_STAT_GRID,
):
    detector = KSWINDetector(
        alpha=alpha,
        window_size=window_size,
        stat_size=stat_size,
        seed=42,
    )

    config = {
        "alpha": alpha,
        "window_size": window_size,
        "stat_size": stat_size,
        "seed": 42,
    }

    rows = evaluate_candidate(
        detector=detector,
        detector_name="kswin",
        config=config,
    )

    search_rows.extend(rows)

    completed += 1

    print(
        f"[{completed}/91] "
        f"KSWIN alpha={alpha}, "
        f"window={window_size}, "
        f"stat={stat_size}"
    )

[8/91] KSWIN alpha=0.0005, window=300, stat=40
[9/91] KSWIN alpha=0.0005, window=300, stat=44
[10/91] KSWIN alpha=0.0005, window=300, stat=48
[11/91] KSWIN alpha=0.0005, window=300, stat=50
[12/91] KSWIN alpha=0.0005, window=300, stat=55
[13/91] KSWIN alpha=0.0005, window=300, stat=60
[14/91] KSWIN alpha=0.0005, window=300, stat=65
[15/91] KSWIN alpha=0.0005, window=450, stat=40
[16/91] KSWIN alpha=0.0005, window=450, stat=44
[17/91] KSWIN alpha=0.0005, window=450, stat=48
[18/91] KSWIN alpha=0.0005, window=450, stat=50
[19/91] KSWIN alpha=0.0005, window=450, stat=55
[20/91] KSWIN alpha=0.0005, window=450, stat=60
[21/91] KSWIN alpha=0.0005, window=450, stat=65
[22/91] KSWIN alpha=0.001, window=300, stat=40
[23/91] KSWIN alpha=0.001, window=300, stat=44
[24/91] KSWIN alpha=0.001, window=300, stat=48
[25/91] KSWIN alpha=0.001, window=300, stat=50
[26/91] KSWIN alpha=0.001, window=300, stat=55
[27/91] KSWIN alpha=0.001, window=300, stat=60
[28/91] KSWIN alpha=0.001, window=300, stat=65
[

In [15]:
search_df = pd.DataFrame(search_rows)

print("Shape:", search_df.shape)
print()
print(search_df.groupby("method").size())

Shape: (1820, 9)

method
adwin     140
kswin    1680
dtype: int64


In [16]:
print("Unique ADWIN configs:",
      search_df[search_df["method"] == "adwin"]["config"].nunique())

print("Unique KSWIN configs:",
      search_df[search_df["method"] == "kswin"]["config"].nunique())

print("Seeds:",
      sorted(search_df["seed"].unique()))

print("Drift types:",
      sorted(search_df["drift_type"].unique()))

Unique ADWIN configs: 7
Unique KSWIN configs: 84
Seeds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Drift types: ['gradual', 'none', 'recurring', 'sudden']


In [17]:
none_clean = (
    search_df[search_df["drift_type"] == "none"]
    .groupby(["method", "config"])["n_false_alarms"]
    .apply(lambda x: bool((x == 0).all()))
    .rename("none_clean")
)

none_clean.head()

method  config            
adwin   {'delta': 0.0001}      True
        {'delta': 0.00025}     True
        {'delta': 0.0005}      True
        {'delta': 0.00075}     True
        {'delta': 0.0015}     False
Name: none_clean, dtype: bool

In [18]:
config_summary = (
    search_df
    .groupby(["method", "config"])
    .agg(
        worst_false_alarms_per_year=("false_alarms_per_year", "max"),
        total_missed_detections=("missed_detections", "sum"),
        mean_detection_delay=("detection_delay", "mean"),
    )
    .join(none_clean)
    .reset_index()
)

config_summary["accepted"] = (
    (config_summary["worst_false_alarms_per_year"] <= FALSE_ALARM_BUDGET_PER_YEAR)
    & (config_summary["total_missed_detections"] == 0)
    & (config_summary["none_clean"])
)

config_summary.head()

,method,config,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_clean,accepted
0,adwin,{'delta': 0.0001},1.752,0,88.766667,True,True
1,adwin,{'delta': 0.00025},1.752,0,86.633333,True,True
2,adwin,{'delta': 0.0005},1.752,0,86.633333,True,True
3,adwin,{'delta': 0.00075},1.752,0,84.500000,True,True
4,adwin,{'delta': 0.0015},4.380,0,84.500000,False,False


In [19]:
config_summary.groupby("method")["accepted"].value_counts()

method  accepted
adwin   True         4
        False        3
kswin   False       80
        True         4
Name: count, dtype: int64

In [20]:
accepted_configs = config_summary[
    config_summary["accepted"]
].copy()

winners = (
    accepted_configs
    .sort_values(
        ["method", "mean_detection_delay"]
    )
    .groupby("method", as_index=False)
    .first()
)

winners

,method,config,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_clean,accepted
0,adwin,{'delta': 0.00075},1.752,0,84.5,True,True
1,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",0.876,0,133.0,True,True


In [21]:
ph_rows = []
ph_completed = 0
ph_total = (
    len(PH_DELTA_GRID)
    * len(PH_MIN_INSTANCES_GRID)
    * len(PH_THRESHOLD_GRID)
)

print("Page-Hinkley configs:", ph_total)
print("Expected runs:", ph_total * len(KINDS) * len(SEEDS))

Page-Hinkley configs: 108
Expected runs: 2160


In [22]:
for delta, min_instances, threshold in product(
    PH_DELTA_GRID,
    PH_MIN_INSTANCES_GRID,
    PH_THRESHOLD_GRID,
):
    detector = PageHinkleyDetector(
        min_instances=min_instances,
        delta=delta,
        threshold=threshold,
    )

    config = {
        "min_instances": min_instances,
        "delta": delta,
        "threshold": threshold,
    }

    rows = evaluate_candidate(
        detector=detector,
        detector_name="page_hinkley",
        config=config,
    )

    ph_rows.extend(rows)

    ph_completed += 1

    print(
        f"[{ph_completed}/{ph_total}] "
        f"Page-Hinkley "
        f"min={min_instances}, "
        f"delta={delta}, "
        f"threshold={threshold}"
    )

[1/108] Page-Hinkley min=20, delta=0.0005, threshold=200.0
[2/108] Page-Hinkley min=20, delta=0.0005, threshold=300.0
[3/108] Page-Hinkley min=20, delta=0.0005, threshold=400.0
[4/108] Page-Hinkley min=20, delta=0.0005, threshold=500.0
[5/108] Page-Hinkley min=20, delta=0.0005, threshold=750.0
[6/108] Page-Hinkley min=20, delta=0.0005, threshold=1000.0
[7/108] Page-Hinkley min=30, delta=0.0005, threshold=200.0
[8/108] Page-Hinkley min=30, delta=0.0005, threshold=300.0
[9/108] Page-Hinkley min=30, delta=0.0005, threshold=400.0
[10/108] Page-Hinkley min=30, delta=0.0005, threshold=500.0
[11/108] Page-Hinkley min=30, delta=0.0005, threshold=750.0
[12/108] Page-Hinkley min=30, delta=0.0005, threshold=1000.0
[13/108] Page-Hinkley min=50, delta=0.0005, threshold=200.0
[14/108] Page-Hinkley min=50, delta=0.0005, threshold=300.0
[15/108] Page-Hinkley min=50, delta=0.0005, threshold=400.0
[16/108] Page-Hinkley min=50, delta=0.0005, threshold=500.0
[17/108] Page-Hinkley min=50, delta=0.0005, thr

In [23]:
ph_df = pd.DataFrame(ph_rows)

print("Shape:", ph_df.shape)
print("Unique configs:", ph_df["config"].nunique())
print("Seeds:", sorted(ph_df["seed"].unique()))
print("Drift types:", sorted(ph_df["drift_type"].unique()))

Shape: (2160, 9)
Unique configs: 108
Seeds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Drift types: ['gradual', 'none', 'recurring', 'sudden']


In [24]:
ph_none_clean = (
    ph_df[ph_df["drift_type"] == "none"]
    .groupby(["method", "config"])["n_false_alarms"]
    .apply(lambda x: bool((x == 0).all()))
    .rename("none_clean")
)

In [25]:
ph_summary = (
    ph_df
    .groupby(["method", "config"])
    .agg(
        worst_false_alarms_per_year=("false_alarms_per_year", "max"),
        total_missed_detections=("missed_detections", "sum"),
        mean_detection_delay=("detection_delay", "mean"),
    )
    .join(ph_none_clean)
    .reset_index()
)

ph_summary["accepted"] = (
    (ph_summary["worst_false_alarms_per_year"] <= FALSE_ALARM_BUDGET_PER_YEAR)
    & (ph_summary["total_missed_detections"] == 0)
    & (ph_summary["none_clean"])
)

ph_summary.head()

,method,config,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_clean,accepted
0,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",0.876,0,232.766667,True,True
1,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",21.900,0,62.766667,False,False
2,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",3.504,0,85.066667,False,False
3,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",1.752,0,113.133333,True,True
4,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",1.752,0,136.400000,True,True


In [26]:
ph_summary["accepted"].value_counts()

accepted
True     72
False    36
Name: count, dtype: int64

In [27]:
ph_winner = (
    ph_summary[ph_summary["accepted"]]
    .sort_values("mean_detection_delay")
    .head(1)
)

ph_winner

,method,config,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_clean,accepted
3,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",1.752,0,113.133333,True,True


In [28]:
print(ph_winner.iloc[0]["config"])

{'min_instances': 20, 'delta': 0.0005, 'threshold': 400.0}


In [56]:
independent_winners = pd.DataFrame(
    [
        {
            "detector": "ADWIN",
            "config": "{'delta': 0.00075}",
            "worst_false_alarms_per_year": 1.752,
            "total_missed_detections": 0,
            "mean_detection_delay": 84.5,
            "none_clean": True,
        },
        {
            "detector": "KSWIN",
            "config": "{'alpha': 0.001, 'window_size': 450, 'stat_size': 55, 'seed': 42}",
            "worst_false_alarms_per_year": 0.876,
            "total_missed_detections": 0,
            "mean_detection_delay": 133.0,
            "none_clean": True,
        },
        {
            "detector": "Page-Hinkley",
            "config": "{'min_instances': 20, 'delta': 0.0005, 'threshold': 400.0}",
            "worst_false_alarms_per_year": 1.752,
            "total_missed_detections": 0,
            "mean_detection_delay": 113.133333,
            "none_clean": True,
        },
    ]
)

independent_winners

,detector,config,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_clean
0,ADWIN,{'delta': 0.00075},1.752,0,84.500000,True
1,KSWIN,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",0.876,0,133.000000,True
2,Page-Hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",1.752,0,113.133333,True


In [57]:
official_winners = pd.read_csv(
    PROJECT_ROOT / "results" / "tables" / "fine_tune_on_synthetic_winners.csv"
)

official_winners

,method,sweep,config,all_runs_accepted,worst_false_alarms_per_year,total_missed_detections,mean_detection_delay,none_total_false_alarms,fallback
0,adwin,adwin_budget_delta,{'delta': 0.00075},True,1.752,0,84.500000,0,False
1,kswin,kswin_budget_grid,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",True,0.876,0,133.000000,0,False
2,page_hinkley,page_hinkley_budget_grid,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",True,1.752,0,113.133333,0,False


In [58]:
independent_winners = pd.DataFrame([
    {
        "method": "adwin",
        "independent_config": "{'delta': 0.00075}",
        "independent_mean_delay": 84.5,
        "independent_worst_fa_per_year": 1.752,
    },
    {
        "method": "kswin",
        "independent_config": "{'alpha': 0.001, 'window_size': 450, 'stat_size': 55, 'seed': 42}",
        "independent_mean_delay": 133.0,
        "independent_worst_fa_per_year": 0.876,
    },
    {
        "method": "page_hinkley",
        "independent_config": "{'min_instances': 20, 'delta': 0.0005, 'threshold': 400.0}",
        "independent_mean_delay": 113.133333,
        "independent_worst_fa_per_year": 1.752,
    },
])

independent_winners

,method,independent_config,independent_mean_delay,independent_worst_fa_per_year
0,adwin,{'delta': 0.00075},84.500000,1.752
1,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",133.000000,0.876
2,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",113.133333,1.752


In [59]:
comparison = independent_winners.merge(
    official_winners[
        [
            "method",
            "config",
            "mean_detection_delay",
            "worst_false_alarms_per_year",
        ]
    ],
    on="method",
    how="left",
)

comparison = comparison.rename(
    columns={
        "config": "official_config",
        "mean_detection_delay": "official_mean_delay",
        "worst_false_alarms_per_year": "official_worst_fa_per_year",
    }
)

comparison

,method,independent_config,independent_mean_delay,independent_worst_fa_per_year,official_config,official_mean_delay,official_worst_fa_per_year
0,adwin,{'delta': 0.00075},84.500000,1.752,{'delta': 0.00075},84.500000,1.752
1,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",133.000000,0.876,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",133.000000,0.876
2,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",113.133333,1.752,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",113.133333,1.752


In [60]:
comparison["config_match"] = (
    comparison["independent_config"]
    == comparison["official_config"]
)

comparison["delay_difference"] = (
    comparison["independent_mean_delay"]
    - comparison["official_mean_delay"]
)

comparison

,method,independent_config,independent_mean_delay,independent_worst_fa_per_year,official_config,official_mean_delay,official_worst_fa_per_year,config_match,delay_difference
0,adwin,{'delta': 0.00075},84.500000,1.752,{'delta': 0.00075},84.500000,1.752,True,0.000000e+00
1,kswin,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",133.000000,0.876,"{'alpha': 0.001, 'window_size': 450, 'stat_siz...",133.000000,0.876,True,0.000000e+00
2,page_hinkley,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",113.133333,1.752,"{'min_instances': 20, 'delta': 0.0005, 'thresh...",113.133333,1.752,True,-3.333333e-07


In [61]:
independent_all = pd.concat(
    [search_df, ph_df],
    ignore_index=True
)

official_all = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "tables"
    / "fine_tune_on_synthetic.csv"
)

print("Independent shape:", independent_all.shape)
print("Official shape:", official_all.shape)
print("Independent columns:", list(independent_all.columns))
print("Official columns:", list(official_all.columns))

Independent shape: (3980, 9)
Official shape: (3980, 13)
Independent columns: ['method', 'config', 'drift_type', 'seed', 'n_detections', 'n_false_alarms', 'false_alarms_per_year', 'detection_delay', 'missed_detections']
Official columns: ['method', 'sweep', 'config', 'drift_type', 'seed', 'n_detections', 'n_false_alarms', 'false_alarms_per_year', 'budget_met', 'detection_delay', 'missed_detections', 'none_clean', 'accepted']


In [62]:
common_cols = [
    "method",
    "config",
    "drift_type",
    "seed",
    "n_detections",
    "n_false_alarms",
    "false_alarms_per_year",
    "detection_delay",
    "missed_detections",
]

sort_cols = [
    "method",
    "config",
    "drift_type",
    "seed",
]

ind_cmp = (
    independent_all[common_cols]
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

off_cmp = (
    official_all[common_cols]
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

print("Same shape:", ind_cmp.shape == off_cmp.shape)

Same shape: True


In [63]:
comparison_check = pd.DataFrame({
    "column": common_cols,
    "all_equal": [
        ind_cmp[col].equals(off_cmp[col])
        for col in common_cols
    ],
})

comparison_check

,column,all_equal
0,method,True
1,config,True
2,drift_type,True
3,seed,True
4,n_detections,True
5,n_false_alarms,True
6,false_alarms_per_year,False
7,detection_delay,True
8,missed_detections,True


In [64]:
import numpy as np

exact_columns = [
    "method",
    "config",
    "drift_type",
    "seed",
    "n_detections",
    "n_false_alarms",
    "missed_detections",
]

numeric_columns = [
    "false_alarms_per_year",
    "detection_delay",
]

for col in exact_columns:
    print(
        col,
        (ind_cmp[col].fillna("__NA__") == off_cmp[col].fillna("__NA__")).all()
    )

for col in numeric_columns:
    print(
        col,
        np.allclose(
            ind_cmp[col].to_numpy(dtype=float),
            off_cmp[col].to_numpy(dtype=float),
            equal_nan=True,
        )
    )

method True
config True
drift_type True
seed True
n_detections True
n_false_alarms True
missed_detections True
false_alarms_per_year True
detection_delay True
